#### Joke Generation in LG with InMemorySaver using persistence

In [29]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [30]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.6, max_tokens=500)

In [31]:
# Create a state graph
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [32]:
# graph
joke_graph = StateGraph(JokeState)

In [33]:
# Nodes
def generate_joke(state: JokeState):
    prompt = f"Generate a joke about {state['topic']}."
    joke = model.invoke(prompt).content
    return {'joke': joke}

def explain_joke(state: JokeState):
    prompt = f"Explain the joke: {state['joke']}."
    explanation = model.invoke(prompt).content
    return {'explanation': explanation}

In [34]:
# define nodes
joke_graph.add_node('generate_joke', generate_joke)
joke_graph.add_node('explain_joke', explain_joke)

In [35]:
# Define edges
joke_graph.add_edge(START, 'generate_joke')
joke_graph.add_edge('generate_joke', 'explain_joke')
joke_graph.add_edge('explain_joke', END)

In [36]:
# Memory creator 
memory = InMemorySaver()

In [37]:
# Compile the graph with a memory saver
workflow = joke_graph.compile(checkpointer=memory)

In [38]:
# Memory related configuration
thread_id = 'joke_workflow_1'
thread_id2 = 'joke_workflow_2'
configuration = {'configurable': {'thread_id': thread_id}}

In [39]:
# Workflow invokation

result = workflow.invoke({'topic': 'cat'}, config=configuration)
print(result)

{'topic': 'cat', 'joke': 'Why did the cat sit on the computer?\n\nBecause it wanted to keep an eye on the mouse!', 'explanation': 'The joke plays on a double meaning of the word "mouse." In the context of computers, a "mouse" is a device used to navigate and interact with the computer interface. However, "mouse" also refers to the small rodent that cats typically chase and hunt. \n\nThe humor comes from the idea that the cat is sitting on the computer not just to be near the device but because it\'s playfully "keeping an eye" on the rodent, suggesting that it is treating the computer mouse as if it were a real mouse. This clever wordplay creates a lighthearted and amusing image of a cat being both curious about technology and instinctively focused on its natural prey.'}


In [40]:
configuration2 = {'configurable': {'thread_id': thread_id2}}
result2 = workflow.invoke({'topic': 'dog'}, config=configuration2)
print(result2)

{'topic': 'dog', 'joke': 'Why did the dog sit in the shade? \n\nBecause he didn’t want to become a hot dog!', 'explanation': 'The joke plays on a pun involving the term "hot dog." In the first part, the question sets up a scenario where a dog is seeking comfort from the heat by sitting in the shade. The punchline, "Because he didn’t want to become a hot dog!" introduces a clever wordplay. \n\n"Hot dog" has two meanings: one refers to a cooked sausage typically served in a bun (a popular food), and the other is a playful term for something being overheated or in distress due to heat. The humor arises from the idea that the dog is trying to avoid becoming a "hot dog" in both senses—by staying cool in the shade, he avoids turning into a literal hot dog (the food) and also avoids overheating. The unexpected twist of the dog\'s concern about his identity adds to the joke\'s charm.'}


In [41]:
workflow.get_state(config=configuration)

StateSnapshot(values={'topic': 'cat', 'joke': 'Why did the cat sit on the computer?\n\nBecause it wanted to keep an eye on the mouse!', 'explanation': 'The joke plays on a double meaning of the word "mouse." In the context of computers, a "mouse" is a device used to navigate and interact with the computer interface. However, "mouse" also refers to the small rodent that cats typically chase and hunt. \n\nThe humor comes from the idea that the cat is sitting on the computer not just to be near the device but because it\'s playfully "keeping an eye" on the rodent, suggesting that it is treating the computer mouse as if it were a real mouse. This clever wordplay creates a lighthearted and amusing image of a cat being both curious about technology and instinctively focused on its natural prey.'}, next=(), config={'configurable': {'thread_id': 'joke_workflow_1', 'checkpoint_ns': '', 'checkpoint_id': '1f18e95a-8232-66ff-8002-02aad48a075c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {

In [42]:
workflow.get_state(config=configuration2)

StateSnapshot(values={'topic': 'dog', 'joke': 'Why did the dog sit in the shade? \n\nBecause he didn’t want to become a hot dog!', 'explanation': 'The joke plays on a pun involving the term "hot dog." In the first part, the question sets up a scenario where a dog is seeking comfort from the heat by sitting in the shade. The punchline, "Because he didn’t want to become a hot dog!" introduces a clever wordplay. \n\n"Hot dog" has two meanings: one refers to a cooked sausage typically served in a bun (a popular food), and the other is a playful term for something being overheated or in distress due to heat. The humor arises from the idea that the dog is trying to avoid becoming a "hot dog" in both senses—by staying cool in the shade, he avoids turning into a literal hot dog (the food) and also avoids overheating. The unexpected twist of the dog\'s concern about his identity adds to the joke\'s charm.'}, next=(), config={'configurable': {'thread_id': 'joke_workflow_2', 'checkpoint_ns': '', 

In [ ]:
workflow.get_state_history(config=configuration)

<generator object Pregel.get_state_history at 0x00000227B453C9E0>